## Evaluacija i poređenje modela

U ovoj svesci evaluiramo i poredimo tri varijante Mini-CLIP modela, koje se razlikuju po image encoder-u.

Istrenirani modeli se nalaze: TODO (videti da li na gitu ili drajvu)

Za svaki model računamo 

Evaluacija se radi na test skupu -  slikama koje model nije video nikad pre.

In [26]:
import os
import re
import json
import ast
from collections import Counter

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F
import torchvision
import torchvision.transforms as transforms
import torchvision.models as models

from PIL import Image

from torch.utils.data import Dataset, DataLoader
from image_encoder import (
    ResNet18Encoder,
    ResNet18EncoderPretrained,
    ResNet18EncoderFineTuned
)

from text_encoder import MiniTextTransformer

### Podešavanje i učitavanje podataka

Test skup učitavamo iz `data/processed/test.csv`.
`test.csv` već sadrži pripremljene `input_ids` i `attention_mask`.
Posle učitavanja iz csv fajla te kolone su stringovi, pa je potrebno da ih parsiramo u Python liste.

In [27]:
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DATA_DIR = "./data"
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
IMAGES_DIR = os.path.join(DATA_DIR, "Images")
MODEL_DIR = "./models" # AKO SU MODELI SKINUTI NA NEKU DRUGU LOKACIJU, OVDE STAVITI PUTANJU

In [28]:
with open(os.path.join(PROCESSED_DIR, "vocab.json"), encoding="utf-8") as f:
    vocab_data = json.load(f)

VOCAB_SIZE = len(vocab_data["token_to_id"])
MAX_LENGTH = vocab_data["max_length"]
PAD_ID = vocab_data["token_to_id"]["<PAD>"]

#test skup - vec preprocesiran u 01_data_setup:
test_df = pd.read_csv(os.path.join(PROCESSED_DIR, "test.csv")).reset_index(drop=True)

In [29]:
def parse_list(value):
    return json.loads(value.replace("'", '"'))

test_df["input_ids"] = test_df["input_ids"].apply(parse_list)
test_df["attention_mask"] = test_df["attention_mask"].apply(parse_list)

#mapiranje za Recall@K
unique_images = test_df["image"].unique()
n_images = len(unique_images)
n_captions = len(test_df)

In [30]:
n_images

810

In [31]:
n_captions

4050

In [32]:
test_df.head()

,image,caption,input_ids,attention_mask
0,1022454428_b6b660a67b.jpg,"a couple and an infant , being held by the mal...","[2, 60, 1615, 221, 216, 3420, 11, 627, 3189, 1...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
1,1022454428_b6b660a67b.jpg,a couple sit on the grass with a baby and stro...,"[2, 60, 1615, 6028, 4476, 6895, 2953, 7625, 60...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, ..."
2,1022454428_b6b660a67b.jpg,a couple with their newborn baby sitting under...,"[2, 60, 1615, 7625, 6897, 4354, 417, 6033, 723...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
3,1022454428_b6b660a67b.jpg,a man and woman care for an infant along the s...,"[2, 60, 3968, 221, 7636, 1111, 2672, 216, 3420...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, ..."
4,1022454428_b6b660a67b.jpg,couple with a baby sit outdoors next to their ...,"[2, 1615, 7625, 60, 417, 6028, 4541, 4361, 698...","[1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 1, 0, 0, ..."


In [35]:
# Isti image transform kao u `01_data_setup.ipynb`
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

image_transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])

## Učitavanje modela

In [46]:
EMB_DIM = 256
K_VALS = [1, 5, 10]

MODEL_CONFIG = [
    { 
        "enc_class": ResNet18Encoder,
        "checkpoint": os.path.join(MODEL_DIR, "best_model.pt"),
        "name": "from scratch"
    },
    { 
        "enc_class": ResNet18EncoderPretrained,
        "checkpoint": os.path.join(MODEL_DIR, "best_model_pretrained.pt"),
        "name": "pretrained"
    },
    { 
        "enc_class": ResNet18EncoderFineTuned,
        "checkpoint": os.path.join(MODEL_DIR, "best_model_fine_tuned.pt"),
        "name": "fine-tuned"
    },
    
]

In [61]:
#loss iz 04
def clip_contrastive_loss(image_embeddings, text_embeddings, temperature=0.07):
    logits = (
        image_embeddings @ text_embeddings.T
    ) / temperature

    labels = torch.arange(logits.size(0), device=logits.device)

    image_to_text_loss = F.cross_entropy(logits, labels)

    text_to_image_loss = F.cross_entropy(logits.T, labels)

    loss = (image_to_text_loss + text_to_image_loss) / 2

    return loss

def build_and_load_model(config):
    image_encoder =  config["enc_class"](emb_dim=EMB_DIM).to(DEVICE)
    text_encoder = MiniTextTransformer(
        vocab_size=VOCAB_SIZE, 
        max_length=MAX_LENGTH, 
        padding_idx=PAD_ID, 
        output_dim=EMB_DIM).to(DEVICE)

    checkpoint = torch.load(config["checkpoint"], map_location=DEVICE)
    image_encoder.load_state_dict(checkpoint["image_encoder_state_dict"])
    text_encoder.load_state_dict(checkpoint["text_encoder_state_dict"])

    return image_encoder, text_encoder, checkpoint
    

In [62]:
@torch.no_grad()
def encode_all_images(image_encoder, batch_size=32):
    image_encoder.eval()
    all_emb = []
    for start in range(0, n_images, batch_size):
        names = unique_images[start:start+batch_size] 
        batch = torch.stack([
                image_transform(Image.open(os.path.join(IMAGES_DIR, n)).convert("RGB")) for n in names
        ]).to(DEVICE)
        all_emb.append(image_encoder(batch).cpu())
    return torch.cat(all_emb, dim=0)

In [63]:
@torch.no_grad()
def encode_all_captions(text_encoder, batch_size=32):
    text_encoder.eval()
    all_emb = []
    for start in range(0, n_captions, batch_size):
        chunk = test_df.iloc[start:start+batch_size]
        input_ids = torch.tensor(chunk["input_ids"].tolist(), dtype=torch.long).to(DEVICE)
        attention_mask = torch.tensor(chunk["attention_mask"].tolist(), dtype=torch.long).to(DEVICE)
        all_emb.append(text_encoder(input_ids, attention_mask).cpu())
    return torch.cat(all_emb, dim=0)

## Metrike

Za svaki model računamo Recall@1 i Recall@5 u oba smera:
- **image -> text**: da li se tačan opis nalazi među top-K najsličnijih caption-a za datu sliku.
- **text -> image**: da li se tačna slika nalazi među top-K najsličnijih slika za dati opis.

TODO razmisliti da li da dodam za jos neko K

In [64]:
def recall_image_to_text(image_emb, text_emb, k):
    sim = image_emb @ text_emb.T
    hits = []
    for img_idx in range(n_images):
        topk = sim[img_idx].topk(k).indices.tolist()
        positives = set(image_to_caption_idxs[img_idx])
        hits.append(any(i in positives for i in topk))

    return float(np.mean(hits))

def recall_text_to_image(image_emb, text_emb, k):
    sim = image_emb @ text_emb.T
    hits = []
    for caption_idx, row in test_df.iterrows():
        img_idx = image_to_idx[row["image"]]
        hits.append(img_idx in sim[caption_idx].topk(k).indices.tolist())

    return float(np.mean(hits))

In [68]:
class TestDataset(Dataset):
    def __len__(self):
        return len(test_df)

    def __getitem__(self, idx):
        row = test_df.iloc[idx]
        img = image_transform(Image.open(os.path.join(IMAGES_DIR, row["image"])).convert("RGB"))
        return (
            img, 
            torch.tensor(row["input_ids"], dtype=torch.long), 
            torch.tensor(row["attention_mask"], dtype=torch.long)
        )
                              

@torch.no_grad()
def compute_test_loss(image_encoder, text_encoder, batch_size=8):
    loader = DataLoader(TestDataset(), batch_size=batch_size, shuffle=False)
    image_encoder.eval()
    text_encoder.eval()

    total = 0.0
    for images, input_ids, attention_mask in loader:
        images = images.to(DEVICE)
        input_ids = input_ids.to(DEVICE)
        attention_mask = attention_mask.to(DEVICE)

        img_emb = image_encoder(images)
        txt_emb = text_encoder(input_ids, attention_mask)
        total += clip_contrastive_loss(img_emb, txt_emb).item()

    return total/len(loader)
    

In [69]:
def evaluate_model(config):
    image_encoder, text_encoder, checkpoint = build_and_load_model(config)
    img_emb = F.normalize(encode_all_images(image_encoder), dim=-1)
    txt_emb = F.normalize(encode_all_captions(text_encoder), dim=-1)

    result = {
        "model": config["name"],
        "epoch": checkpoint.get("epoch", -1),
        "val_loss": float(checkpoint.get("val_loss", float("nan"))),
        "test_loss": compute_test_loss(image_encoder, text_encoder)
    }

    for k in K_VALS:
        result[f"image2text_R@{k}"] = recall_image_to_text(image_emb, text_emb, k)
        result[f"text2image_R@{k}"] = recall_text_to_image(image_emb, text_emb, k)

    return result
        
    

## Evaluacija svih modela

Za svaki checkpoint učitavamo modele i računamo metrike na test skupu

In [ ]:
results = []
for config in MODEL_CONFIG:
    print(f" evaluating model {config['name']}")
    res = evaluate_model(config)
    results.append(res)

results_df = pd.DataFrame(results)
results_df.round(4)

 evaluating model from scratch
